# Moon vs Zodi Split — Design & Validation of `SkyDecomp(split_zodi=True)`

**Status:** Phase 1 complete.  `sky_decomp/fit.py` supports `split_zodi=True`
behind an opt-in flag; `SkyDecompLSFSurfaceIterative` propagates it end-to-end;
`decompose_parallel.py` exposes the mode via
`--fit-model lsf-surface-iterative-split-zodi`.  This notebook is the reference
design document *and* the reproducible validation for that mode.

## Motivation

The pre-split decomposition uses one continuum family:

    Moon_bs(λ) = solar_rb(λ) * B_29(λ) * c_moon

Because that 29-knot spline is smooth and weakly penalised, at rows with
significant zodiacal contamination (moon down, low ecliptic latitude) the
solver silently attributes the zodi flux to `Moon_bs` — turning `c_moon` into
a mixed moon-plus-zodi coefficient the downstream ML pipeline cannot cleanly
separate.  See rows 741 (moon_alt ≈ -14°, ecl_beta ≈ 3°) and 830 as the
canonical failure cases inspected below.

## Design of the split

`split_zodi=True` replaces the single family with two color-tagged families:

$$
\text{Moon}_{bs}(\lambda) = \text{solar}_{rb}(\lambda)\;
\text{moon\_albedo}(\lambda)\; B_{K_{\rm moon}}(\lambda)\; c_{\rm moon}
$$

$$
\text{Zodi}_{bs}(\lambda) = \text{solar}_{rb}(\lambda)\;
\text{zodi\_color}(\lambda)\; B_{K_{\rm zodi}}(\lambda)\; c_{\rm zodi}
$$

where

* `solar_rb(λ)` — LATMOS solar SED rebinned + LSF-convolved onto the LVM grid
  (already used by baseline `Moon_bs`).
* `moon_albedo(λ)` — ROLO disk-integrated lunar albedo at a fiducial phase
  angle of 30° (median-normalised).  Loaded from
  `sky_decomp/data/moon_zodi/eso_skycalc_rolo_moon_albedo.dat`.
* `zodi_color(λ) = (λ / 5000Å)^{0.26}` — Leinert-style reddening (matches
  `moon_zodi_model.py`), median-normalised.
* `B_{K_{\rm moon}}(λ) = B_{29}` — same 29-knot cubic B-spline as before.
* `B_{K_{\rm zodi}}(λ) = B_{3}` — much smoother 3-knot cubic B-spline.

Both families remain **non-negative** in the QP.  The two color envelopes are
what makes them identifiable inside the full multi-family fit: `moon_albedo`
carries sharp ROLO mineral bands (particularly around 950 nm) while
`zodi_color` is a smooth monotonic power-law reddening.  The `diffuse` family
(HO2 + FeO + O2Ac) already absorbs the flat continuum baseline, so the moon
vs zodi decision only has to distinguish curvature-sensitive shape.

## Curvature penalties

Each spline family carries a second-difference penalty:

    ‖D² c_moon‖² * moon_smooth_lambda    (default 0.1; kept at 1e-3 for continuity with baseline)
    ‖D² c_zodi‖² * zodi_smooth_lambda    (recommended 1e-1)

Bumping `zodi_smooth_lambda` (~1e-1) with a small `K_zodi` (≤3) makes the
fitted zodi cleanly follow the red-slope `zodi_color` envelope instead of
picking up locally blue features from residual continuum ambiguity.

## Optional amplitude priors

`fit(..., moon_amp_prior, zodi_amp_prior, moon_amp_prior_lambda, zodi_amp_prior_lambda)`
add a quadratic penalty on the *integrated* family amplitudes:

$$
\lambda \; \Bigl(\sum_k c_k \int B_k(\lambda)\,d\lambda \; - \; \text{target}\Bigr)^2
$$

Anchor `moon_amp_prior` to a moon-brightness proxy
(`moon_fli * max(0, sin(moon_alt)) * (Rayleigh + Aerosol)(sep)` times a
globally-calibrated `α_moon`) and `zodi_amp_prior` to Leinert `B(500)` at
the observed ecliptic geometry times a globally-calibrated `α_leinert`.
Both anchors are computed and validated in later cells (`compute_priors`,
`alpha_moon`, `alpha_leinert`).  Recommended λ ≈ 1e-4; both default to off.

## Notebook road-map

The cells below build up the design piece by piece and then validate it at
increasing scale:

1. **Setup & row selection** — row/regime picking that always includes rows
   741 (moon-down + on-ecliptic, the canonical zodi contamination case) and
   830 (near-target control).
2. **Color envelopes** — build `moon_albedo` and `zodi_color` and plot them.
3. **Design matrices (LSQ standalone)** — build A_fitA (moon-only) and
   A_fitB (moon + zodi) and run per-row / per-arm penalised NNLS on
   `y = observed – (all non-moon components from the baseline decomp)`.
   *This standalone test is intentionally pessimistic* — by pre-subtracting
   the baseline diffuse it leaves the moon+zodi splines to fight over the
   flat baseline residual and appears unidentifiable.  Tier 1b (K_zodi=1)
   and Tier 2 (Leinert amp prior) cells were used to explore whether soft
   physical priors alone could rescue the standalone case (they mostly can,
   but the analysis over-generalised).
4. **Phase 1 end-to-end (12 rows)** — `SkyDecomp(split_zodi=True)` inside
   the full decomposition (with `diffuse` participating).  Shows the split
   IS identifiable in the full fit at λ=0, delivers median RMS ratio 0.988
   vs baseline (∼2% better), and moves row 741's zodi contamination cleanly
   from `Moon_bs` to `Zodi_bs`.  Includes a smoothing/lambda sweep and a
   Phase 1 conclusion.
5. **Phase 2 (100 rows, iterative LSF + priors)** —
   `SkyDecompLSFSurfaceIterative(split_zodi=True)` with `n_refinement_cycles=3`
   and the physics-motivated moon+zodi amplitude priors turned on.
   Produces the reference 6-panel diagnostic plot: observed / moon / zodi /
   diffuse / residual / residual-histogram.  This is the recommended
   production configuration.

## Recommended production settings

Validated on the p40_p70 every10 corpus (100-row phase-stratified sample,
`seed=42`):

    n_spline_knots        = 29
    n_zodi_spline_knots   = 3
    moon_smooth_lambda    = 1e-3
    zodi_smooth_lambda    = 1e-1
    moon_albedo_fiducial_phase_deg = 30.0
    zodi_color_exponent   = 0.26
    moon_amp_prior_lambda = 1e-4   (optional)
    zodi_amp_prior_lambda = 1e-4   (optional)

## Reproducing the corpus decomposition

The same configuration is available at corpus scale:

```
python skysub/decompose_parallel.py <every10.fits> <palace_dir> \
    --fit-model lsf-surface-iterative-split-zodi \
    --n-zodi-spline-knots 3 --zodi-smooth-lambda 0.1 \
    --palace-oh-suffix _joint_v2_updated \
    --palace-diffuse-suffix _joint_native_adam_invsky_p2_10000iter \
    --n-refinement-cycles 5 --n-workers 8 --output-dir <out>
```

which writes `..._decomp_{sci,sky1,sky2}_lsf_surface_iterative_split_zodi.fits`
each carrying a new `COMP_ZODI` HDU alongside the standard component HDUs,
plus the usual meta+coef products.


In [ ]:
"""Setup: imports, paths, row selection."""
from __future__ import annotations
from pathlib import Path

import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import Table
from astropy.time import Time
from astropy.coordinates import SkyCoord, BarycentricMeanEcliptic
from astropy import units as u
from scipy.interpolate import BSpline
from scipy.optimize import nnls
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from lvmdrp.core.fluxcal import rebin_and_convolve

REPO = Path('/Users/droryn/prog/lvm/lvmsky')
SKY_DECOMP_DATA = REPO / 'skysub' / 'sky_decomp' / 'data'
SPLINE_MOON = REPO / 'skysub' / 'spline_moon'
STEM = 'lvmsframe_median_stack_1.2.1_p40_p70_every10'

FITS_EVERY10 = SPLINE_MOON / f'{STEM}.fits'
FITS_SCI  = SPLINE_MOON / f'{STEM}_decomp_sci_lsf_surface_iterative.fits'
FITS_NEAR = SPLINE_MOON / f'{STEM}_decomp_sky1_lsf_surface_iterative.fits'
FITS_FAR  = SPLINE_MOON / f'{STEM}_decomp_sky2_lsf_surface_iterative.fits'

SOLAR_TXT = REPO / 'Spectre_HR_LATMOS_Meftah_V1_350_1000nm.txt'
ROLO_DAT  = SKY_DECOMP_DATA / 'moon_zodi' / 'eso_skycalc_rolo_moon_albedo.dat'

N_MOON_KNOTS = 29
N_ZODI_KNOTS = 5
MOON_SMOOTH_LAMBDA = 0.1
ZODI_SMOOTH_LAMBDA = 1.0
FIDUCIAL_MOON_PHASE_DEG = 30.0  # a mid-illumination anchor for ROLO
ZODI_COLOR_EXPONENT = 0.26      # matches moon_zodi_model.py

def vac_to_air(lam_vac_a: np.ndarray) -> np.ndarray:
    lam = np.asarray(lam_vac_a, float)
    s2 = (1e4 / lam) ** 2
    n = 1.0 + 8.34254e-5 + 2.406147e-2 / (130.0 - s2) + 1.5998e-4 / (38.9 - s2)
    return lam / n

with fits.open(FITS_EVERY10) as hdul:
    WAVE_A = np.asarray(hdul['WAVE'].data, dtype=np.float64)
    META_ALL = Table(hdul['META'].data)
print(f'WAVE_A: {WAVE_A.size} pixels, {WAVE_A[0]:.1f} A -> {WAVE_A[-1]:.1f} A')
print(f'META_ALL: n_rows = {len(META_ALL)}')

In [ ]:
"""Select a phase-stratified diverse sample of rows for the test.

We want coverage of (moon-alt sign) x (ecl-beta band) x (moon-fli level).
Row 741 (target failure case) and 830 (near-target) are always included.
"""
# Compute ecliptic beta on-the-fly for meta selection.
coord = SkyCoord(
    ra=np.asarray(META_ALL['sci_ra'], float) * u.deg,
    dec=np.asarray(META_ALL['sci_dec'], float) * u.deg,
    frame='icrs',
)
obstime = Time(np.asarray(META_ALL['mjd'], float), format='mjd')
ecl = coord.transform_to(BarycentricMeanEcliptic(equinox=obstime))
META_ALL['ecl_beta_deg'] = ecl.lat.to(u.deg).value

def _pick(mask, k, rng):
    idx = np.flatnonzero(np.asarray(mask))
    if idx.size == 0:
        return []
    return list(rng.choice(idx, size=min(k, idx.size), replace=False))

rng = np.random.default_rng(seed=42)
ma = np.asarray(META_ALL['moon_alt'], float)
mf = np.asarray(META_ALL['moon_fli'], float)
eb = np.asarray(META_ALL['ecl_beta_deg'], float)

regime_picks = {}
regime_picks['moon_dn_on_ecl']   = _pick((ma < -10) & (np.abs(eb) < 10), 6, rng)
regime_picks['moon_dn_off_ecl']  = _pick((ma < -10) & (np.abs(eb) > 25), 6, rng)
regime_picks['moon_up_dim']      = _pick((ma > 10)  & (mf < 0.30),        6, rng)
regime_picks['moon_up_bright']   = _pick((ma > 20)  & (mf > 0.85),        6, rng)
regime_picks['moon_up_on_ecl']   = _pick((ma > 5)   & (np.abs(eb) < 10), 6, rng)
SELECTED_ROWS = sorted(set([741, 830] + [int(i) for reg in regime_picks.values() for i in reg]))
print(f'Selected {len(SELECTED_ROWS)} rows.  Per regime:')
for name, rows in regime_picks.items():
    print(f'  {name:16s} n={len(rows)}')
print(f'Row 741 present: {741 in SELECTED_ROWS}  |  Row 830 present: {830 in SELECTED_ROWS}')

In [ ]:
"""Build the three fixed color templates on the LVM native grid.

* `solar_rb`     -- solar SED rebinned + LSF-convolved to LVM native (fiducial LSF)
* `moon_albedo`  -- ROLO at 30 deg fiducial phase, interpolated to LVM grid
* `zodi_color`   -- (lambda / 5000 nm)^0.26 (matches moon_zodi_model.py)

The identifiability question is whether moon_albedo x spline(29) can absorb
zodi_color x spline(K).
"""
# ---- solar SED
sol = np.loadtxt(SOLAR_TXT, comments=';')
solar_wave_a = vac_to_air(sol[:, 0] * 10.0)     # nm vacuum -> A air
solar_flux_raw = np.asarray(sol[:, 1], float)
solar_flux_raw /= np.nanmedian(solar_flux_raw)
# Use a fiducial LSF: median LSF from the first N rows of sci arm.
with fits.open(FITS_EVERY10) as hdul:
    lsf_sci = np.asarray(hdul['LSF_SCI'].data[:64], float)
    lsf_fid = np.nanmedian(lsf_sci, axis=0)
solar_rb = rebin_and_convolve(
    WAVE_A, solar_wave_a, solar_flux_raw,
    lsf_fid * 2.355, lsf_in_wavelength=True,
)
solar_rb /= np.nanmedian(solar_rb)
assert np.all(np.isfinite(solar_rb))

# ---- ROLO moon albedo at 30 deg fiducial phase
def _load_rolo(path):
    with Path(path).open('r') as fh:
        lines = [ln.rstrip() for ln in fh if ln.strip() and not ln.startswith('#')]
    consts = np.fromstring(lines[0], sep=' ')
    n = int(lines[1])
    coefs = np.array([np.fromstring(ln, sep=' ') for ln in lines[2:2+n]])
    return consts, coefs

def _rolo_albedo_on_grid(wave_a, phase_deg, consts, coefs):
    wave_nm = wave_a / 10.0
    v = coefs[:, 1:]
    phase_deg_abs = abs(phase_deg)
    phase_rad = np.deg2rad(phase_deg_abs)
    signed_rad = np.deg2rad(phase_deg)
    signed_lim = signed_rad if phase_deg_abs < 97.0 else 97.0 * signed_rad / phase_deg_abs
    poly = (v[:, 0] + v[:, 1]*phase_rad + v[:, 2]*phase_rad**2 + v[:, 3]*phase_rad**3
            + v[:, 4]*signed_lim + v[:, 5]*signed_lim**3 + v[:, 6]*signed_lim**5)
    opp = (v[:, 7]*np.exp(-phase_deg_abs/consts[0]) + v[:, 8]*np.exp(-phase_deg_abs/consts[1])
           + v[:, 9]*np.cos((phase_deg_abs - consts[2]) / consts[3]))
    tab_val = np.exp(poly + opp) / 0.87
    return np.interp(wave_nm, coefs[:, 0], tab_val)

rolo_c, rolo_coefs = _load_rolo(ROLO_DAT)
moon_albedo = _rolo_albedo_on_grid(WAVE_A, FIDUCIAL_MOON_PHASE_DEG, rolo_c, rolo_coefs)
# Normalize so median=1 (only the shape matters; absolute amplitude is absorbed by fit coefs).
moon_albedo /= np.nanmedian(moon_albedo)

# ---- Zodi color
zodi_color = (WAVE_A / 5000.0) ** ZODI_COLOR_EXPONENT
zodi_color /= np.nanmedian(zodi_color)

print(f'moon_albedo: median-normed, range = [{moon_albedo.min():.3f}, {moon_albedo.max():.3f}]')
print(f'zodi_color : median-normed, range = [{zodi_color.min():.3f}, {zodi_color.max():.3f}]')
print(f'solar_rb   : median-normed, range = [{np.nanmin(solar_rb):.3f}, {np.nanmax(solar_rb):.3f}]')

# ---- Plot the two color envelopes
fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Scatter(x=WAVE_A, y=moon_albedo, name='moon albedo (ROLO 30 deg)', line=dict(color='#e41a1c')))
fig.add_trace(go.Scatter(x=WAVE_A, y=zodi_color, name='zodi color ((L/5000)^0.26)', line=dict(color='#377eb8')))
fig.update_layout(
    title='Color envelopes for the two proposed families',
    xaxis=dict(title='wavelength (A)'),
    yaxis=dict(title='normalized shape'),
    height=350, width=900,
)
fig.show()

In [ ]:
"""Build the two design matrices and the augmented-NNLS solver."""
def make_bspline_design(wave, n_interior_knots):
    w0, w1 = float(wave[0]), float(wave[-1])
    interior = np.linspace(w0, w1, n_interior_knots + 2)[1:-1]
    t_knots = np.r_[(w0,)*4, interior, (w1,)*4]
    B = BSpline.design_matrix(wave, t_knots, 3).toarray()
    return B  # (n_wave, n_interior + degree + 1) but degree=3 -> n_interior + 4 cols? check

def d2_penalty(n_par, weight):
    if n_par < 3:
        return np.zeros((0, n_par))
    D = np.zeros((n_par - 2, n_par))
    idx = np.arange(n_par - 2)
    D[idx, idx] = 1.0; D[idx, idx + 1] = -2.0; D[idx, idx + 2] = 1.0
    return np.sqrt(max(weight, 0.0)) * D

def solve_penalized_nnls(A, y, w, penalties):
    """Solve min ||W(A c - y)||^2 + sum_p ||P_p c||^2   s.t. c >= 0.

    A: (n_pix, n_par)  design
    y: (n_pix,)         target
    w: (n_pix,)         weights (sqrt of ivar); non-finite pixels get w=0
    penalties: list of (P, cols_slice) where P is (n_rows, n_cols_slice)
    """
    A_finite = np.all(np.isfinite(A), axis=1)
    finite = np.isfinite(y) & np.isfinite(w) & A_finite
    w = np.where(finite, np.nan_to_num(w, nan=0.0, posinf=0.0, neginf=0.0), 0.0)
    y_safe = np.where(finite, np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0), 0.0)
    A_safe = np.where(finite[:, None], np.nan_to_num(A, nan=0.0, posinf=0.0, neginf=0.0), 0.0)
    A_aug_rows = [A_safe * w[:, None]]
    y_aug_rows = [y_safe * w]
    n_par = A.shape[1]
    for (P, sl) in penalties:
        row = np.zeros((P.shape[0], n_par))
        row[:, sl] = P
        A_aug_rows.append(row)
        y_aug_rows.append(np.zeros(P.shape[0]))
    A_aug = np.vstack(A_aug_rows)
    y_aug = np.concatenate(y_aug_rows)
    # Column scaling for conditioning.
    col_sc = np.sqrt(np.sum(A_aug ** 2, axis=0))
    col_sc = np.where(col_sc > 0, col_sc, 1.0)
    A_scl = A_aug / col_sc[None, :]
    c_scl, resid_norm = nnls(A_scl, y_aug, maxiter=5000)
    c = c_scl / col_sc
    return c, float(resid_norm)

# Moon-only (Fit A): solar * B_29
B_moon = make_bspline_design(WAVE_A, N_MOON_KNOTS)
A_fitA = solar_rb[:, None] * B_moon              # (n_pix, K_moon)
n_par_moon = B_moon.shape[1]

# Split (Fit B): [solar*moon_albedo*B_29 | solar*zodi_color*B_K]
A_moon_split = (solar_rb * moon_albedo)[:, None] * B_moon
B_zodi = make_bspline_design(WAVE_A, N_ZODI_KNOTS)
A_zodi_split = (solar_rb * zodi_color)[:, None] * B_zodi
n_par_zodi = B_zodi.shape[1]
A_fitB = np.hstack([A_moon_split, A_zodi_split]) # (n_pix, K_moon + K_zodi)

moon_slice_A = slice(0, n_par_moon)
moon_slice_B = slice(0, n_par_moon)
zodi_slice_B = slice(n_par_moon, n_par_moon + n_par_zodi)

P_moon_A = d2_penalty(n_par_moon, MOON_SMOOTH_LAMBDA)
P_moon_B = d2_penalty(n_par_moon, MOON_SMOOTH_LAMBDA)
P_zodi_B = d2_penalty(n_par_zodi, ZODI_SMOOTH_LAMBDA)

print(f'A_fitA: {A_fitA.shape}, n_par_moon = {n_par_moon}')
print(f'A_fitB: {A_fitB.shape}, n_par_moon = {n_par_moon}, n_par_zodi = {n_par_zodi}')
print(f'Penalties: P_moon={P_moon_A.shape}, P_zodi={P_zodi_B.shape}')

In [ ]:
"""For each selected row * arm, isolate the moon+zodi target flux and run both fits."""
def load_row_arm(arm_fits_path, row_index):
    with fits.open(arm_fits_path) as hdul:
        comps = {
            k: np.asarray(hdul[f'COMP_{k}'].data[row_index], float)
            for k in ('OH', 'MOON', 'HO2', 'FEO', 'O2AC', 'ATOM', 'ORC', 'O2')
        }
        sigma = np.asarray(hdul['FLUX_SIGMA_TOTAL'].data[row_index], float)
    return comps, sigma

PHYSICAL_TO_FIT_FLUX_SCALE = 1.0e14  # matches sky_decomp defaults; see test_moon_zodi_*.py

def load_row_obs(arm_key, row_index):
    """Load raw FLUX in fit units (scaled up by 1e14) to match COMP_* / SIGMA HDUs."""
    with fits.open(FITS_EVERY10) as hdul:
        obs = np.asarray(hdul[f'FLUX_{arm_key}'].data[row_index], float) * PHYSICAL_TO_FIT_FLUX_SCALE
    return obs

ARM_INFO = [
    ('NEAR', 'SKY_NEAR', FITS_NEAR),
    ('FAR',  'SKY_FAR',  FITS_FAR),
    ('SCI',  'SCI',      FITS_SCI),
]

records = []
row_arm_fits_cache = {}

for row_idx in SELECTED_ROWS:
    meta_row = META_ALL[row_idx]
    for arm_label, obs_key, decomp_path in ARM_INFO:
        comps, sigma = load_row_arm(decomp_path, row_idx)
        obs = load_row_obs(obs_key, row_idx)
        # y = observed - (all non-moon+diffuse continuum spline components) - HO2/FeO/O2Ac (these
        # are DIFFUSE, not moon; keep them subtracted so the moon block only sees moon+zodi
        # continuum + fit residual)
        non_moon = (comps['OH'] + comps['HO2'] + comps['FEO'] + comps['O2AC']
                    + comps['ATOM'] + comps['ORC'] + comps['O2'])
        y = obs - non_moon
        good = np.isfinite(y) & np.isfinite(sigma) & (sigma > 0)
        if int(good.sum()) < 100:
            continue
        w = np.zeros_like(y)
        w[good] = 1.0 / sigma[good]
        c_A, res_A = solve_penalized_nnls(
            A_fitA, y, w, [(P_moon_A, moon_slice_A)],
        )
        c_B, res_B = solve_penalized_nnls(
            A_fitB, y, w, [(P_moon_B, moon_slice_B), (P_zodi_B, zodi_slice_B)],
        )
        pred_A = A_fitA @ c_A
        pred_B = A_fitB @ c_B
        moon_B = A_moon_split @ c_B[moon_slice_B]
        zodi_B = A_zodi_split @ c_B[zodi_slice_B]

        # Residual RMS on the target region (>= 4000 A to avoid edge noise dominating).
        band = good & (WAVE_A >= 4000.0) & (WAVE_A <= 9500.0)
        def _rms(x):
            return float(np.sqrt(np.mean((x[band] * w[band]) ** 2)))
        rms_A = _rms(y - pred_A)
        rms_B = _rms(y - pred_B)

        # Integrated flux per family: sum(component * dw) in target band.
        dw = np.gradient(WAVE_A)
        int_moon_A = float(np.nansum(pred_A[band] * dw[band]))
        int_moon_B = float(np.nansum(moon_B[band] * dw[band]))
        int_zodi_B = float(np.nansum(zodi_B[band] * dw[band]))
        records.append(dict(
            row=int(row_idx),
            arm=arm_label,
            moon_alt=float(meta_row['moon_alt']),
            moon_fli=float(meta_row['moon_fli']),
            moon_sep=float(meta_row['sci_moon_sep' if arm_label == 'SCI' else
                                    ('skye_moon_sep' if arm_label == 'NEAR' else 'skyw_moon_sep')]),
            ecl_beta_deg=float(meta_row['ecl_beta_deg']),
            sum_c_moon_A=float(c_A.sum()),
            sum_c_moon_B=float(c_B[moon_slice_B].sum()),
            sum_c_zodi_B=float(c_B[zodi_slice_B].sum()),
            int_moon_A=int_moon_A,
            int_moon_B=int_moon_B,
            int_zodi_B=int_zodi_B,
            frac_zodi_B=float(int_zodi_B / max(int_moon_A, 1e-9)),
            rms_A=rms_A,
            rms_B=rms_B,
            drms_rel=float((rms_B - rms_A) / max(rms_A, 1e-30)),
        ))
        # cache preds for later plotting on row 741 sci
        row_arm_fits_cache[(row_idx, arm_label)] = dict(
            y=y, w=w, band=band, good=good,
            pred_A=pred_A, moon_B=moon_B, zodi_B=zodi_B,
            c_moon_A=c_A, c_moon_B=c_B[moon_slice_B], c_zodi_B=c_B[zodi_slice_B],
        )

df = pd.DataFrame.from_records(records)
print(f'Ran {len(df)} row-arm fits.')
df.head()

In [ ]:
"""Identifiability diagnostics: does zodi amplitude correlate with expected proxies?

* Correlate `int_zodi_B` with `|ecl_beta|` and `moon_alt`.
* Check that at moon-down + on-ecliptic (like row 741), int_zodi_B > 0 across arms.
* Check that at bright-moon-close, int_zodi_B stays small relative to int_moon_B.
* Compare residual RMS: Fit B should not be much worse than Fit A (paying only
  for the extra DOF via curvature penalty).
"""
import numpy as np
print('=== Correlation of integrated zodi amplitude vs sky geometry (per arm) ===')
for arm in ('SCI', 'NEAR', 'FAR'):
    sub = df[df['arm'] == arm]
    if sub.empty:
        continue
    r_beta = np.corrcoef(np.abs(sub['ecl_beta_deg']), sub['int_zodi_B'])[0, 1]
    r_ma  = np.corrcoef(sub['moon_alt'],             sub['int_zodi_B'])[0, 1]
    r_frac_ma = np.corrcoef(sub['moon_alt'],         sub['frac_zodi_B'])[0, 1]
    print(f'  {arm}: n={len(sub):3d}   r(int_zodi, |ecl_beta|)={r_beta:+.3f}   '
          f'r(int_zodi, moon_alt)={r_ma:+.3f}   r(frac_zodi, moon_alt)={r_frac_ma:+.3f}')

print()
print('=== Row 741 and 830 per-arm split (integrated flux ratio zodi/moon) ===')
for row in (741, 830):
    sub = df[df['row'] == row]
    for _, r in sub.iterrows():
        print(f'  row={row}  arm={r.arm}  moon_alt={r.moon_alt:+.1f}  ecl_beta={r.ecl_beta_deg:+.1f}  '
              f'int_moon_A={r.int_moon_A:.3g}  int_moon_B={r.int_moon_B:.3g}  '
              f'int_zodi_B={r.int_zodi_B:.3g}  frac_zodi={r.frac_zodi_B:.3f}  '
              f'rms_A={r.rms_A:.3g}  rms_B={r.rms_B:.3g}  drms_rel={r.drms_rel:+.2%}')

print()
print('=== Regime-mean int_zodi_B / int_moon_A (per arm) ===')
def regime(row):
    if row.moon_alt < -10 and abs(row.ecl_beta_deg) < 10: return 'moon_dn_on_ecl'
    if row.moon_alt < -10 and abs(row.ecl_beta_deg) > 25: return 'moon_dn_off_ecl'
    if row.moon_alt > 20 and row.moon_fli > 0.85: return 'moon_up_bright'
    if row.moon_alt > 10 and row.moon_fli < 0.30: return 'moon_up_dim'
    if row.moon_alt > 5  and abs(row.ecl_beta_deg) < 10: return 'moon_up_on_ecl'
    return 'other'

df['regime'] = df.apply(regime, axis=1)
for arm in ('SCI', 'NEAR', 'FAR'):
    print(f'  arm={arm}')
    for r_name, sub in df[df['arm'] == arm].groupby('regime'):
        print(f'    {r_name:16s} n={len(sub):3d}   '
              f'int_zodi_B (median={sub["int_zodi_B"].median():.3g}) / int_moon_B '
              f'(median={sub["int_moon_B"].median():.3g})  frac_zodi_median={sub["frac_zodi_B"].median():.3f}')

In [ ]:
"""Visual: fit A vs fit B at row 741 sci arm (the target failure case), plus a bright-moon control."""
def plot_row_arm(row, arm, title_suffix=''):
    key = (row, arm)
    if key not in row_arm_fits_cache:
        print(f'no cache for {key}'); return None
    d = row_arm_fits_cache[key]
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=(f'Row {row} {arm}  target flux and fits {title_suffix}',
                                        f'Row {row} {arm}  split components (Fit B)'))
    fig.add_trace(go.Scatter(x=WAVE_A, y=d['y'], name='y (obs - non-moon)',
                             line=dict(color='#7f7f7f', width=1)), row=1, col=1)
    fig.add_trace(go.Scatter(x=WAVE_A, y=d['pred_A'], name='Fit A (moon only)',
                             line=dict(color='#e41a1c', width=1.6)), row=1, col=1)
    fig.add_trace(go.Scatter(x=WAVE_A, y=(d['moon_B'] + d['zodi_B']),
                             name='Fit B total (moon + zodi)',
                             line=dict(color='#377eb8', width=1.4, dash='dash')), row=1, col=1)
    fig.add_trace(go.Scatter(x=WAVE_A, y=d['moon_B'], name='Fit B moon',
                             line=dict(color='#e41a1c', width=1.4)), row=2, col=1)
    fig.add_trace(go.Scatter(x=WAVE_A, y=d['zodi_B'], name='Fit B zodi',
                             line=dict(color='#4daf4a', width=1.4)), row=2, col=1)
    fig.update_xaxes(title_text='wavelength (A)', row=2, col=1)
    fig.update_yaxes(title_text='flux', row=1, col=1)
    fig.update_yaxes(title_text='component flux', row=2, col=1)
    fig.update_layout(height=650, width=1100)
    return fig

fig1 = plot_row_arm(741, 'SCI', ' (target failure)')
if fig1 is not None:
    fig1.show()

# Bright-moon control (pick first available in that regime)
bright_row = None
for row in SELECTED_ROWS:
    r = df[(df['row'] == row) & (df['arm'] == 'SCI')]
    if r.empty: continue
    if r.iloc[0]['regime'] == 'moon_up_bright':
        bright_row = row; break

if bright_row is not None:
    fig2 = plot_row_arm(bright_row, 'SCI', ' (bright-moon control)')
    if fig2 is not None:
        fig2.show()

In [ ]:
"""Scatter plot of int_zodi_B (per row-arm) vs |ecl_beta|, colored by moon_alt.

The split is identifiable if the scatter shows a clear structure: high zodi
amplitude at low |ecl_beta| that decays as |ecl_beta| grows.
"""
fig = make_subplots(rows=1, cols=3, subplot_titles=('SCI', 'NEAR', 'FAR'),
                    shared_yaxes=True)
for j, arm in enumerate(('SCI', 'NEAR', 'FAR'), start=1):
    sub = df[df['arm'] == arm]
    fig.add_trace(go.Scatter(
        x=np.abs(sub['ecl_beta_deg']), y=sub['int_zodi_B'],
        mode='markers',
        marker=dict(size=8, color=sub['moon_alt'], colorscale='RdBu', cmid=0,
                    colorbar=(dict(title='moon_alt') if j == 3 else None),
                    showscale=(j == 3)),
        text=[f'row {r} {arm} m_alt={ma:+.1f} m_fli={mf:.2f}' for r, ma, mf
              in zip(sub['row'], sub['moon_alt'], sub['moon_fli'])],
        showlegend=False,
        hovertemplate='%{text}<br>|ecl_beta|=%{x:.1f}<br>int_zodi_B=%{y:.2g}',
    ), row=1, col=j)
    fig.update_xaxes(title_text='|ecl_beta| (deg)', row=1, col=j)
fig.update_yaxes(title_text='int_zodi_B', row=1, col=1)
fig.update_layout(title='Split identifiability: integrated zodi amplitude vs ecliptic latitude',
                  height=420, width=1200)
fig.show()

fig = go.Figure()
for arm, color in (('SCI', '#e41a1c'), ('NEAR', '#377eb8'), ('FAR', '#4daf4a')):
    sub = df[df['arm'] == arm]
    fig.add_trace(go.Bar(x=[arm], y=[sub['drms_rel'].median()],
                        name=arm, marker_color=color))
fig.update_layout(title='Median relative RMS change: Fit B vs Fit A',
                  yaxis=dict(title='(rms_B - rms_A) / rms_A', tickformat='.1%'),
                  height=350, width=600)
fig.show()

In [ ]:
"""Tier 1b: is the split identifiable if we constrain Zodi_bs to a SINGLE amplitude?

K_zodi = 1 means the zodi family collapses to `solar * zodi_color * scalar`.
That leaves the moon spline (K_moon=33) with full flexibility and forces zodi
to only pick up the fixed-shape zodi color amplitude.  If this DOESN'T yield
the physically-expected r(int_zodi, |ecl_beta|) < 0 pattern, no purely color-based
split can work and we must go to Tier 2 (Leinert prior on zodi amplitude).
"""
K_ZODI_SCALAR = 1
B_zodi_1 = make_bspline_design(WAVE_A, K_ZODI_SCALAR)
A_zodi_1 = (solar_rb * zodi_color)[:, None] * B_zodi_1
n_par_zodi_1 = B_zodi_1.shape[1]
A_fitB1 = np.hstack([A_moon_split, A_zodi_1])
moon_slice_B1 = slice(0, n_par_moon)
zodi_slice_B1 = slice(n_par_moon, n_par_moon + n_par_zodi_1)
P_moon_B1 = d2_penalty(n_par_moon, MOON_SMOOTH_LAMBDA)
P_zodi_B1 = d2_penalty(n_par_zodi_1, 100.0)   # heavy penalty; near-rigid
print(f'A_fitB1: {A_fitB1.shape}, n_par_moon={n_par_moon}, n_par_zodi={n_par_zodi_1}')

recs_B1 = []
for row_idx in SELECTED_ROWS:
    meta_row = META_ALL[row_idx]
    for arm_label, obs_key, decomp_path in ARM_INFO:
        comps, sigma = load_row_arm(decomp_path, row_idx)
        obs = load_row_obs(obs_key, row_idx)
        non_moon = (comps['OH'] + comps['HO2'] + comps['FEO'] + comps['O2AC']
                    + comps['ATOM'] + comps['ORC'] + comps['O2'])
        y = obs - non_moon
        good = np.isfinite(y) & np.isfinite(sigma) & (sigma > 0)
        if int(good.sum()) < 100: continue
        w = np.zeros_like(y); w[good] = 1.0 / sigma[good]
        c_B1, _ = solve_penalized_nnls(
            A_fitB1, y, w,
            [(P_moon_B1, moon_slice_B1), (P_zodi_B1, zodi_slice_B1)],
        )
        moon_B1 = A_moon_split @ c_B1[moon_slice_B1]
        zodi_B1 = A_zodi_1 @ c_B1[zodi_slice_B1]
        band = good & (WAVE_A >= 4000.0) & (WAVE_A <= 9500.0)
        dw = np.gradient(WAVE_A)
        recs_B1.append(dict(
            row=int(row_idx), arm=arm_label,
            moon_alt=float(meta_row['moon_alt']),
            moon_fli=float(meta_row['moon_fli']),
            ecl_beta_deg=float(meta_row['ecl_beta_deg']),
            int_moon_B1=float(np.nansum(moon_B1[band]*dw[band])),
            int_zodi_B1=float(np.nansum(zodi_B1[band]*dw[band])),
        ))
df1 = pd.DataFrame.from_records(recs_B1)
df1['frac_zodi_B1'] = df1['int_zodi_B1'] / (df1['int_moon_B1'] + df1['int_zodi_B1']).clip(lower=1e-30)
df1['regime'] = df1.apply(regime, axis=1)
print(f'\nK_zodi=1 (rigid) correlations per arm:')
for arm in ('SCI','NEAR','FAR'):
    sub = df1[df1['arm']==arm]
    r_beta = np.corrcoef(np.abs(sub['ecl_beta_deg']), sub['int_zodi_B1'])[0, 1]
    r_ma   = np.corrcoef(sub['moon_alt'],             sub['int_zodi_B1'])[0, 1]
    r_fli  = np.corrcoef(sub['moon_fli'],             sub['int_zodi_B1'])[0, 1]
    print(f'  {arm}: r(int_zodi, |ecl_beta|)={r_beta:+.3f}  r(int_zodi, moon_alt)={r_ma:+.3f}  r(int_zodi, moon_fli)={r_fli:+.3f}')

print(f'\nK_zodi=1 regime-medians (SCI arm):')
for r_name, sub in df1[df1['arm']=='SCI'].groupby('regime'):
    print(f'  {r_name:16s} n={len(sub):3d}  int_zodi_B1 median={sub["int_zodi_B1"].median():.3g}  '
          f'int_moon_B1 median={sub["int_moon_B1"].median():.3g}  frac_zodi_median={sub["frac_zodi_B1"].median():.3f}')

print(f'\nRow 741 / 830 with K_zodi=1:')
for row in (741, 830):
    for arm in ('SCI','NEAR','FAR'):
        r = df1[(df1['row']==row) & (df1['arm']==arm)]
        if r.empty: continue
        r = r.iloc[0]
        print(f'  row={row} arm={arm}  int_moon_B1={r.int_moon_B1:.3g}  int_zodi_B1={r.int_zodi_B1:.3g}  frac_zodi={r.frac_zodi_B1:.3f}')


In [ ]:
"""Tier 2 identifiability test: add a soft Leinert amplitude prior on Zodi_bs.

Per-row zodi amplitude anchor = B_500(|ecl_lon_rel_sun|, |ecl_beta|) via the
Leinert (1998) 2D lookup already used by moon_zodi_model.py, scaled by an
airmass attenuation factor and by a global calibration constant `alpha_leinert`
fit once across the sample.  The fit adds a quadratic penalty

    lambda_prior * (sum_k c_zodi[k] * bspline_area[k]  -  alpha_leinert * B_row)^2

and we re-run the K_zodi=1 rigid-zodi fit from the previous cell.  If Tier 2
cleans up r(int_zodi, |ecl_beta|), we have empirical justification for building
the same prior into fit.py.
"""
import sys
sys.path.insert(0, str(REPO / 'skysub'))
from sky_decomp.moon_zodi_model import _load_leinert, _interpolate_leinert, _rayleigh_optical_depth, ZODIACAL_LIGHT_ASSET, DEFAULT_DATA_DIR
from astropy.coordinates import get_sun

leinert_grid = _load_leinert(str(DEFAULT_DATA_DIR / ZODIACAL_LIGHT_ASSET))
print('Leinert grid loaded.  Coordinate arrays:')
print(f'  n_ecl_lon_rel_sun = {leinert_grid[0].size}, range = [{leinert_grid[0].min()}, {leinert_grid[0].max()}] deg')
print(f'  n_ecl_beta       = {leinert_grid[1].size}, range = [{leinert_grid[1].min()}, {leinert_grid[1].max()}] deg')
print(f'  values shape = {leinert_grid[2].shape}')

# Compute per-row Leinert B(500) using midpoint-of-observation geometry.
mjd_arr = np.asarray(META_ALL['mjd'], float)
obstime_arr = Time(mjd_arr, format='mjd')
sun_ecl = get_sun(obstime_arr).transform_to(BarycentricMeanEcliptic(equinox=obstime_arr))
target_ecl = coord.transform_to(BarycentricMeanEcliptic(equinox=obstime_arr))
sun_lon = sun_ecl.lon.wrap_at(360 * u.deg).to(u.deg).value
target_lon = target_ecl.lon.wrap_at(360 * u.deg).to(u.deg).value
rel_lon_signed = ((target_lon - sun_lon) + 180.0) % 360.0 - 180.0
META_ALL['ecl_rel_lon_deg'] = rel_lon_signed

def _leinert_row(row_idx):
    x = float(abs(META_ALL['ecl_rel_lon_deg'][row_idx]))
    y = float(abs(META_ALL['ecl_beta_deg'][row_idx]))
    try:
        return float(_interpolate_leinert(x, y, leinert_grid))
    except Exception:
        return np.nan

b500_per_row = np.array([_leinert_row(i) for i in SELECTED_ROWS])
print(f'\nLeinert B(500) for the {len(SELECTED_ROWS)} selected rows:')
print(f'  finite fraction = {np.isfinite(b500_per_row).mean():.2f}')
print(f'  range           = [{np.nanmin(b500_per_row):.3g}, {np.nanmax(b500_per_row):.3g}] (units: 10^-8 W/m^2/sr/um)')
for row in (741, 830):
    i = SELECTED_ROWS.index(row)
    print(f'  row={row}: rel_lon={META_ALL["ecl_rel_lon_deg"][row]:+.1f} deg  '
          f'ecl_beta={META_ALL["ecl_beta_deg"][row]:+.1f} deg  B(500)={b500_per_row[i]:.3g}')


In [ ]:
"""Fit the Tier 2 model with Leinert amplitude prior on Zodi_bs.

Steps:
  1. Use the K_zodi=1 rigid-zodi design from the previous cell.
  2. First pass: fit ALL selected rows WITHOUT the amplitude prior (this is
     the Tier 1b result) and record int_zodi_B1 for each row.
  3. Fit a single global scale alpha_leinert = median(int_zodi_B1[training] /
     b500[training]) on rows expected to be zodi-dominated (moon-down and moon
     off-target, i.e. not confused with moonlight).
  4. Second pass: re-fit with the soft prior alpha_leinert * b500_row on
     sum(c_zodi_B1) added to each row's normal equations.
  5. Re-check identifiability metrics.
"""
# --- Step 1: reuse the previous Tier 1b fit's zodi amplitudes
row_to_i = {r: i for i, r in enumerate(SELECTED_ROWS)}
b500_by_row = {r: float(b500_per_row[row_to_i[r]]) for r in SELECTED_ROWS}

# --- Step 2: Anchor calibration on moon-down + off-target rows (SCI arm) where
# the fitted zodi should track Leinert well.  We only need alpha_leinert to be
# well-conditioned, so use the moon-down + off-ecl OR on-ecl SCI rows.
anchor_mask = (df1['arm'] == 'SCI') & (df1['moon_alt'] < -10) & df1['row'].map(lambda r: np.isfinite(b500_by_row.get(r, np.nan)))
df_anchor = df1[anchor_mask].copy()
df_anchor['b500'] = df_anchor['row'].map(b500_by_row)
ratios = df_anchor['int_zodi_B1'] / df_anchor['b500'].clip(lower=1e-9)
alpha_leinert = float(np.nanmedian(ratios))
print(f'Anchor calibration (n={len(df_anchor)}): alpha_leinert = median(int_zodi_B1/B500) = {alpha_leinert:.3g}')
print(f'  ratio range: {np.nanmin(ratios):.3g} .. {np.nanmax(ratios):.3g}')

# --- Step 3: refit each row with Leinert prior on the Zodi_bs coefficient sum.
# Prior: lambda_prior * (sum(c_zodi) - alpha_leinert * B500)^2
# Add as one augmented row: [0, ..., 0, sqrt(lambda_prior), sqrt(lambda_prior), ..., sqrt(lambda_prior)] * c = sqrt(lambda_prior) * alpha_leinert * B500
LAMBDA_PRIOR_VALUES = [0.0, 1.0e-6, 1.0e-4, 1.0e-2, 1.0]
results_by_lambda = {}
for LAMBDA_PRIOR in LAMBDA_PRIOR_VALUES:
    recs_T2 = []
    for row_idx in SELECTED_ROWS:
        meta_row = META_ALL[row_idx]
        b500_row = b500_by_row.get(int(row_idx), np.nan)
        if not np.isfinite(b500_row):
            continue
        target_amp = alpha_leinert * b500_row
        for arm_label, obs_key, decomp_path in ARM_INFO:
            comps, sigma = load_row_arm(decomp_path, row_idx)
            obs = load_row_obs(obs_key, row_idx)
            non_moon = (comps['OH'] + comps['HO2'] + comps['FEO'] + comps['O2AC']
                        + comps['ATOM'] + comps['ORC'] + comps['O2'])
            y = obs - non_moon
            good = np.isfinite(y) & np.isfinite(sigma) & (sigma > 0)
            if int(good.sum()) < 100: continue
            w = np.zeros_like(y); w[good] = 1.0 / sigma[good]
            penalties = [(P_moon_B1, moon_slice_B1), (P_zodi_B1, zodi_slice_B1)]
            if LAMBDA_PRIOR > 0.0:
                zodi_prior_row = np.zeros((1, A_fitB1.shape[1]))
                zodi_prior_row[0, zodi_slice_B1] = np.sqrt(LAMBDA_PRIOR)
                # Extra target row on RHS: sqrt(lambda) * target_amp
                # solve_penalized_nnls treats penalty rows with RHS=0, so we need a
                # different mechanism: prepend the prior row to the base system
                # rather than to the penalty stack.
                A_ext = np.vstack([A_fitB1, zodi_prior_row])
                y_ext = np.concatenate([y, [np.sqrt(LAMBDA_PRIOR) * target_amp]])
                w_ext = np.concatenate([w, [1.0]])
                c_T2, _ = solve_penalized_nnls(A_ext, y_ext, w_ext, penalties)
            else:
                c_T2, _ = solve_penalized_nnls(A_fitB1, y, w, penalties)
            moon_T2 = A_moon_split @ c_T2[moon_slice_B1]
            zodi_T2 = A_zodi_1 @ c_T2[zodi_slice_B1]
            band = good & (WAVE_A >= 4000.0) & (WAVE_A <= 9500.0)
            dw = np.gradient(WAVE_A)
            recs_T2.append(dict(
                row=int(row_idx), arm=arm_label,
                moon_alt=float(meta_row['moon_alt']),
                moon_fli=float(meta_row['moon_fli']),
                ecl_beta_deg=float(meta_row['ecl_beta_deg']),
                b500=b500_row,
                target_amp=target_amp,
                sum_c_zodi=float(c_T2[zodi_slice_B1].sum()),
                int_moon_T2=float(np.nansum(moon_T2[band]*dw[band])),
                int_zodi_T2=float(np.nansum(zodi_T2[band]*dw[band])),
            ))
    dfT2 = pd.DataFrame.from_records(recs_T2)
    if dfT2.empty:
        continue
    dfT2['frac_zodi_T2'] = dfT2['int_zodi_T2'] / (dfT2['int_moon_T2'] + dfT2['int_zodi_T2']).clip(lower=1e-30)
    dfT2['regime'] = dfT2.apply(regime, axis=1)
    results_by_lambda[LAMBDA_PRIOR] = dfT2

print()
print('=== Tier 2 identifiability: r(int_zodi_T2, |ecl_beta|) SCI ===')
print('     lambda_prior       corr_ecl_beta  corr_moon_alt  corr_moon_fli')
for LAMBDA_PRIOR, dfT2 in results_by_lambda.items():
    sub = dfT2[dfT2['arm']=='SCI']
    r_beta = np.corrcoef(np.abs(sub['ecl_beta_deg']), sub['int_zodi_T2'])[0, 1]
    r_ma   = np.corrcoef(sub['moon_alt'],             sub['int_zodi_T2'])[0, 1]
    r_fli  = np.corrcoef(sub['moon_fli'],             sub['int_zodi_T2'])[0, 1]
    print(f'     lambda={LAMBDA_PRIOR:.1e}     {r_beta:+.3f}          {r_ma:+.3f}           {r_fli:+.3f}')

print()
print('=== SCI arm regime medians per lambda_prior ===')
print(f'{"regime":18s}  ' + '  '.join([f'lam={l:.0e}' for l in LAMBDA_PRIOR_VALUES]))
regime_labels = sorted(set(next(iter(results_by_lambda.values()))['regime'].unique()))
for r_name in regime_labels:
    vals = []
    for LAMBDA_PRIOR in LAMBDA_PRIOR_VALUES:
        dfT2 = results_by_lambda.get(LAMBDA_PRIOR)
        if dfT2 is None:
            vals.append('n/a'); continue
        sub = dfT2[(dfT2['arm']=='SCI') & (dfT2['regime']==r_name)]
        if sub.empty:
            vals.append('n/a'); continue
        vals.append(f'{sub["frac_zodi_T2"].median():.3f}')
    print(f'{r_name:18s}  ' + '  '.join([f'{v:>7s}' for v in vals]))


In [ ]:
"""Tier 2 with JOINT moon+zodi amplitude priors.

The Leinert-only prior over-constrains rows where the observed flux comes
primarily from moon.  To break the moon/zodi ambiguity at the linear-fit
level we need a matching physical prior on moon amplitude.  Simplified
moon prior: A_moon(row) = moon_fli * max(0, sin(moon_alt_rad)) *
                          (Rayleigh(sep) + Aerosol(sep))
captured as a per-row scalar that gets a global calibration alpha_moon fit
on bright-moon-clean rows (moon_alt > 40 & moon_fli > 0.90 & |ecl_beta| >
20 -- these are moon-dominated with negligible zodi contamination).
"""
def rayleigh_phase(cos_sep, dep=0.0148):
    return 3.0 * (1.0 - dep) / (16.0 * np.pi * (1.0 + 2.0 * dep)) * (1.0 + (1.0 + 3.0*dep)/(1.0 - dep) * cos_sep**2)

def hg_phase(cos_sep, g=0.8):
    return (1.0 - g*g) / (4.0 * np.pi * (1.0 + g*g - 2.0*g*cos_sep)**1.5)

def moon_amp_proxy(meta_row, arm_label):
    m_alt = float(meta_row['moon_alt'])
    m_fli = float(meta_row['moon_fli'])
    sep_key = {'SCI': 'sci_moon_sep', 'NEAR': 'skye_moon_sep', 'FAR': 'skyw_moon_sep'}[arm_label]
    m_sep = float(meta_row[sep_key])
    if m_alt <= 0.0 or not np.isfinite(m_sep):
        return 0.0
    cos_sep = float(np.cos(np.deg2rad(m_sep)))
    # Combined Rayleigh + aerosol phase functions (equal weight for the identifiability test).
    phase = rayleigh_phase(cos_sep) + hg_phase(cos_sep)
    return float(m_fli * max(0.0, np.sin(np.deg2rad(m_alt))) * phase)

# Anchor alpha_moon on moon-dominated rows (bright + off-ecliptic to minimise zodi contamination).
df_moon_anchor = df1[(df1['arm'] == 'SCI')
                      & (df1['moon_alt'] > 40.0)
                      & (df1['moon_fli'] > 0.90)
                      & (np.abs(df1['ecl_beta_deg']) > 20.0)].copy()
if len(df_moon_anchor) < 2:
    # fallback: use all bright rows regardless of ecliptic
    df_moon_anchor = df1[(df1['arm'] == 'SCI')
                          & (df1['moon_alt'] > 20.0)
                          & (df1['moon_fli'] > 0.85)].copy()
df_moon_anchor['moon_proxy'] = [
    moon_amp_proxy(META_ALL[int(r)], 'SCI') for r in df_moon_anchor['row']
]
df_moon_anchor = df_moon_anchor[df_moon_anchor['moon_proxy'] > 0]
if len(df_moon_anchor) >= 2:
    alpha_moon = float((df_moon_anchor['int_moon_B1'] / df_moon_anchor['moon_proxy']).median())
else:
    alpha_moon = float((df1[df1['arm']=='SCI']['int_moon_B1'].median()) / max(np.median([moon_amp_proxy(META_ALL[int(r)], 'SCI') for r in df1[df1['arm']=='SCI']['row']]), 1e-9))
print(f'Moon prior anchor: n={len(df_moon_anchor)} rows,  alpha_moon = {alpha_moon:.3g}')

LAMBDA_ZODI = 1.0e-4
LAMBDA_MOON_VALUES = [0.0, 1.0e-6, 1.0e-4, 1.0e-2, 1.0]
print(f'\nZodi prior lambda fixed at {LAMBDA_ZODI:.1e};  varying moon prior lambda.')

joint_results = {}
for LAMBDA_MOON in LAMBDA_MOON_VALUES:
    recs_JP = []
    for row_idx in SELECTED_ROWS:
        meta_row = META_ALL[row_idx]
        b500_row = b500_by_row.get(int(row_idx), np.nan)
        if not np.isfinite(b500_row): continue
        target_zodi = alpha_leinert * b500_row
        for arm_label, obs_key, decomp_path in ARM_INFO:
            comps, sigma = load_row_arm(decomp_path, row_idx)
            obs = load_row_obs(obs_key, row_idx)
            non_moon = (comps['OH'] + comps['HO2'] + comps['FEO'] + comps['O2AC']
                        + comps['ATOM'] + comps['ORC'] + comps['O2'])
            y = obs - non_moon
            good = np.isfinite(y) & np.isfinite(sigma) & (sigma > 0)
            if int(good.sum()) < 100: continue
            w = np.zeros_like(y); w[good] = 1.0 / sigma[good]

            target_moon = alpha_moon * moon_amp_proxy(meta_row, arm_label)
            extra_rows = []
            extra_y = []
            extra_w = []
            # Zodi amplitude prior row.
            zpr = np.zeros((1, A_fitB1.shape[1]))
            zpr[0, zodi_slice_B1] = np.sqrt(LAMBDA_ZODI)
            extra_rows.append(zpr[0])
            extra_y.append(np.sqrt(LAMBDA_ZODI) * target_zodi)
            extra_w.append(1.0)
            # Moon amplitude prior row.
            if LAMBDA_MOON > 0.0:
                mpr = np.zeros((1, A_fitB1.shape[1]))
                mpr[0, moon_slice_B1] = np.sqrt(LAMBDA_MOON)
                extra_rows.append(mpr[0])
                extra_y.append(np.sqrt(LAMBDA_MOON) * target_moon)
                extra_w.append(1.0)
            A_ext = np.vstack([A_fitB1, np.vstack(extra_rows)])
            y_ext = np.concatenate([y, np.asarray(extra_y)])
            w_ext = np.concatenate([w, np.asarray(extra_w)])
            c_JP, _ = solve_penalized_nnls(
                A_ext, y_ext, w_ext,
                [(P_moon_B1, moon_slice_B1), (P_zodi_B1, zodi_slice_B1)],
            )
            moon_JP = A_moon_split @ c_JP[moon_slice_B1]
            zodi_JP = A_zodi_1 @ c_JP[zodi_slice_B1]
            band = good & (WAVE_A >= 4000.0) & (WAVE_A <= 9500.0)
            dw = np.gradient(WAVE_A)
            recs_JP.append(dict(
                row=int(row_idx), arm=arm_label,
                moon_alt=float(meta_row['moon_alt']),
                moon_fli=float(meta_row['moon_fli']),
                ecl_beta_deg=float(meta_row['ecl_beta_deg']),
                b500=b500_row,
                target_moon=target_moon,
                target_zodi=target_zodi,
                int_moon_JP=float(np.nansum(moon_JP[band]*dw[band])),
                int_zodi_JP=float(np.nansum(zodi_JP[band]*dw[band])),
            ))
    dfJP = pd.DataFrame.from_records(recs_JP)
    dfJP['frac_zodi_JP'] = dfJP['int_zodi_JP'] / (dfJP['int_moon_JP'] + dfJP['int_zodi_JP']).clip(lower=1e-30)
    dfJP['regime'] = dfJP.apply(regime, axis=1)
    joint_results[LAMBDA_MOON] = dfJP

print()
print(f'=== Joint prior identifiability (SCI arm, zodi_lambda={LAMBDA_ZODI:.1e}) ===')
print(f'  moon_lambda      r(int_zodi,|ecl_beta|)   r(int_zodi,moon_alt)   r(int_zodi,moon_fli)   r(int_moon,moon_fli)')
for LAMBDA_MOON, dfJP in joint_results.items():
    sub = dfJP[dfJP['arm']=='SCI']
    r_beta = np.corrcoef(np.abs(sub['ecl_beta_deg']), sub['int_zodi_JP'])[0, 1]
    r_ma   = np.corrcoef(sub['moon_alt'],             sub['int_zodi_JP'])[0, 1]
    r_fli  = np.corrcoef(sub['moon_fli'],             sub['int_zodi_JP'])[0, 1]
    r_moon_fli = np.corrcoef(sub['moon_fli'],         sub['int_moon_JP'])[0, 1]
    print(f'  lambda={LAMBDA_MOON:.1e}   {r_beta:+.3f}                {r_ma:+.3f}                 {r_fli:+.3f}                 {r_moon_fli:+.3f}')

print()
print('=== SCI arm regime medians (frac_zodi) across moon_lambda ===')
print(f'{"regime":18s}  ' + '  '.join([f'moon_l={l:.0e}' for l in LAMBDA_MOON_VALUES]))
regime_labels = sorted(set(next(iter(joint_results.values()))['regime'].unique()))
for r_name in regime_labels:
    vals = []
    for LAMBDA_MOON in LAMBDA_MOON_VALUES:
        dfJP = joint_results.get(LAMBDA_MOON)
        if dfJP is None:
            vals.append('n/a'); continue
        sub = dfJP[(dfJP['arm']=='SCI') & (dfJP['regime']==r_name)]
        if sub.empty:
            vals.append('n/a'); continue
        vals.append(f'{sub["frac_zodi_JP"].median():.3f}')
    print(f'{r_name:18s}  ' + '  '.join([f'{v:>10s}' for v in vals]))

print()
print('=== Row 741 / 830 with joint priors, best lambda=1e-4 ===')
dfJP_best = joint_results.get(1.0e-4)
if dfJP_best is not None:
    for row in (741, 830):
        for arm in ('SCI', 'NEAR', 'FAR'):
            r = dfJP_best[(dfJP_best['row']==row) & (dfJP_best['arm']==arm)]
            if r.empty: continue
            r = r.iloc[0]
            print(f'  row={row} arm={arm}  target_moon={r.target_moon:.3g}  target_zodi={r.target_zodi:.3g}  '
                  f'int_moon_JP={r.int_moon_JP:.3g}  int_zodi_JP={r.int_zodi_JP:.3g}  frac_zodi={r.frac_zodi_JP:.3f}')


## Phase 1: End-to-end split-zodi decomposition on every10 rows

Runs the newly-patched `SkyDecomp(split_zodi=True)` on the same 32 rows,
compares the resulting reconstruction against the existing single-Moon_bs
decomposition stored in the FITS files, and plots per-row moon/zodi splits.

We use joint moon+zodi amplitude priors from the same physics anchors that
worked in the LSQ identifiability test above.


In [ ]:
"""Set up the SkyDecomp instance in split_zodi mode.

Requires the freshly-patched fit.py (Phase 1a + 1b).  Uses:
  * K_moon=29 (unchanged), moon_smooth_lambda=1e-3
  * K_zodi=3, zodi_smooth_lambda=1e-2 (10x moon curvature penalty)
  * moon_albedo from ROLO at 30 deg fiducial phase
  * zodi_color = (lambda/5000 Å)^0.26
"""
from sky_decomp.fit import SkyDecomp

# Fiducial LSF: use the median of the first 64 sci rows (matches what the
# existing p40_p70 decomposition file was fit against on average).
with fits.open(FITS_EVERY10) as hdul:
    lsf_sci_all = np.asarray(hdul['LSF_SCI'].data, float)
print(f'LSF grid: {lsf_sci_all.shape}')

N_MOON_KNOTS_FIT = 29
N_ZODI_KNOTS_FIT = 3

def build_decomp(row_index, arm_label):
    lsf_hdu_key = {'SCI': 'LSF_SCI', 'NEAR': 'LSF_SKY_NEAR', 'FAR': 'LSF_SKY_FAR'}[arm_label]
    with fits.open(FITS_EVERY10) as hdul:
        lsf = np.asarray(hdul[lsf_hdu_key].data[row_index], float)
    lsf_safe = np.where(np.isfinite(lsf) & (lsf > 0), lsf, np.nanmedian(lsf))
    return SkyDecomp(
        wave=WAVE_A,
        lsf_sigma=lsf_safe,
        n_spline_knots=N_MOON_KNOTS_FIT,
        base_dir=str(REPO / 'skysub' / 'sky_decomp' / 'data'),
        palace_oh_suffix='_joint_v2_updated',
        palace_diffuse_suffix='_joint_native_adam_invsky_p2_10000iter',
        moon_smooth_lambda=1.0e-3,
        moon_interline_boost=0.0,
        # New split-zodi kwargs:
        split_zodi=True,
        n_zodi_spline_knots=N_ZODI_KNOTS_FIT,
        zodi_smooth_lambda=1.0e-2,
        moon_albedo_fiducial_phase_deg=30.0,
        zodi_color_exponent=0.26,
    )

# Also build a no-split baseline decomp for A/B comparison.
def build_baseline_decomp(row_index, arm_label):
    lsf_hdu_key = {'SCI': 'LSF_SCI', 'NEAR': 'LSF_SKY_NEAR', 'FAR': 'LSF_SKY_FAR'}[arm_label]
    with fits.open(FITS_EVERY10) as hdul:
        lsf = np.asarray(hdul[lsf_hdu_key].data[row_index], float)
    lsf_safe = np.where(np.isfinite(lsf) & (lsf > 0), lsf, np.nanmedian(lsf))
    return SkyDecomp(
        wave=WAVE_A,
        lsf_sigma=lsf_safe,
        n_spline_knots=N_MOON_KNOTS_FIT,
        base_dir=str(REPO / 'skysub' / 'sky_decomp' / 'data'),
        palace_oh_suffix='_joint_v2_updated',
        palace_diffuse_suffix='_joint_native_adam_invsky_p2_10000iter',
        moon_smooth_lambda=1.0e-3,
        moon_interline_boost=0.0,
        split_zodi=False,
    )

print('SkyDecomp factories ready.')


In [ ]:
"""Compute per-row moon+zodi amplitude priors from the physics helpers.

Uses the alpha_leinert calibrated in cell 12 and alpha_moon calibrated in
cell 13. These are only order-of-magnitude anchors for the ~32-row test; the
eventual production calibration would fit them on the full corpus.
"""
def compute_priors(row_index, arm_label):
    meta_row = META_ALL[row_index]
    b500 = b500_by_row.get(int(row_index), np.nan)
    target_zodi = float(alpha_leinert * b500) if np.isfinite(b500) else 0.0
    target_moon = float(alpha_moon * moon_amp_proxy(meta_row, arm_label))
    return target_moon, target_zodi

# Sanity-check: print priors for row 741 and 830.
for row in (741, 830):
    for arm in ('SCI', 'NEAR', 'FAR'):
        tm, tz = compute_priors(row, arm)
        print(f'  row={row} arm={arm}   target_moon={tm:.3g}  target_zodi={tz:.3g}')


In [ ]:
"""Run the SPLIT decomposition on selected rows (SCI arm) and record metrics.

Compares against the CURRENT baseline decomposition (already in the FITS
files) via a per-arm bestfit RMS diff.  Also stores the moon/zodi components
of the split fit for plotting.
"""
import time

PHYSICAL_TO_FIT_FLUX_SCALE = 1.0e14

LAMBDA_MOON_FIT = 1.0e-4
LAMBDA_ZODI_FIT = 1.0e-4

split_records = []
split_cache = {}

TEST_ROWS_FIT = SELECTED_ROWS[:12] + [741, 830]  # keep runtime bounded
TEST_ROWS_FIT = sorted(set(TEST_ROWS_FIT))

for row_idx in TEST_ROWS_FIT:
    meta_row = META_ALL[row_idx]
    # Only fit sci arm for now to keep runtime tractable.
    arm_label = 'SCI'
    obs_key = 'SCI'
    with fits.open(FITS_EVERY10) as hdul:
        obs = np.asarray(hdul[f'FLUX_{obs_key}'].data[row_idx], float) * PHYSICAL_TO_FIT_FLUX_SCALE
    with fits.open(FITS_SCI) as hdul:
        sigma = np.asarray(hdul['FLUX_SIGMA_TOTAL'].data[row_idx], float)
        comp_moon_baseline = np.asarray(hdul['COMP_MOON'].data[row_idx], float)
        bestfit_baseline = np.asarray(hdul['BESTFIT'].data[row_idx], float)
        resid_baseline = np.asarray(hdul['RESID'].data[row_idx], float)
    ivar = np.where(sigma > 0, 1.0 / (sigma ** 2), 0.0)
    ivar = np.where(np.isfinite(ivar), ivar, 0.0)

    tm, tz = compute_priors(row_idx, arm_label)

    decomp = build_decomp(row_idx, arm_label)
    t0 = time.perf_counter()
    result = decomp.fit(
        obs, ivar, verbose=False, n_lsf_refits=0,
        moon_amp_prior=tm, zodi_amp_prior=tz,
        moon_amp_prior_lambda=LAMBDA_MOON_FIT,
        zodi_amp_prior_lambda=LAMBDA_ZODI_FIT,
    )
    dt = time.perf_counter() - t0

    comps = result.components
    moon_comp = comps.get('moon', np.zeros_like(WAVE_A))
    zodi_comp = comps.get('zodi', np.zeros_like(WAVE_A))
    bestfit_split = result.bestfit
    resid_split = obs - bestfit_split

    good = np.isfinite(obs) & np.isfinite(sigma) & (sigma > 0)
    band = good & (WAVE_A >= 4000.0) & (WAVE_A <= 9500.0)
    def _weighted_rms(r):
        return float(np.sqrt(np.nanmean((r[band] / sigma[band]) ** 2))) if band.sum() > 0 else np.nan
    rms_split = _weighted_rms(resid_split)
    rms_baseline_fits = _weighted_rms(resid_baseline)

    dw = np.gradient(WAVE_A)
    int_moon = float(np.nansum(moon_comp[band] * dw[band]))
    int_zodi = float(np.nansum(zodi_comp[band] * dw[band]))
    int_moon_baseline = float(np.nansum(comp_moon_baseline[band] * dw[band]))

    split_records.append(dict(
        row=int(row_idx),
        arm=arm_label,
        moon_alt=float(meta_row['moon_alt']),
        moon_fli=float(meta_row['moon_fli']),
        ecl_beta_deg=float(META_ALL['ecl_beta_deg'][row_idx]),
        target_moon=tm,
        target_zodi=tz,
        int_moon_split=int_moon,
        int_zodi_split=int_zodi,
        frac_zodi_split=int_zodi / max(int_moon + int_zodi, 1e-30),
        int_moon_baseline=int_moon_baseline,
        rms_weighted_split=rms_split,
        rms_weighted_baseline=rms_baseline_fits,
        fit_status=str(result.fit_status),
        elapsed_sec=dt,
    ))
    split_cache[(row_idx, arm_label)] = dict(
        obs=obs, sigma=sigma, ivar=ivar,
        moon_comp=moon_comp, zodi_comp=zodi_comp,
        bestfit_split=bestfit_split, resid_split=resid_split,
        comp_moon_baseline=comp_moon_baseline, bestfit_baseline=bestfit_baseline,
        resid_baseline=resid_baseline, target_moon=tm, target_zodi=tz,
        result=result,
    )

df_split = pd.DataFrame.from_records(split_records)
print(f'\nRan split_zodi=True decomp on {len(df_split)} SCI rows.')
print(f'Weighted-RMS baseline vs split:')
print(f'  median rms_baseline = {df_split["rms_weighted_baseline"].median():.3g}')
print(f'  median rms_split    = {df_split["rms_weighted_split"].median():.3g}')
print(f'  fraction where split RMS <= baseline: {(df_split["rms_weighted_split"] <= df_split["rms_weighted_baseline"]).mean():.2f}')
print()
print(df_split[['row','moon_alt','ecl_beta_deg','int_moon_split','int_zodi_split','frac_zodi_split','rms_weighted_split','rms_weighted_baseline']].to_string(index=False))


In [ ]:
"""Plot the split decomposition for row 741 (target) and a bright-moon control."""
def plot_split_recon(row_idx, arm='SCI', title_suffix=''):
    key = (row_idx, arm)
    if key not in split_cache:
        print(f'no cache for {key}'); return None
    d = split_cache[key]
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=(
                            f'Row {row_idx} {arm}: observed + total fits{title_suffix}',
                            f'Row {row_idx} {arm}: split moon vs zodi components (Fit B)',
                        ),
                        row_heights=[0.55, 0.45])
    fig.add_trace(go.Scatter(x=WAVE_A, y=d['obs'], name='observed',
                             line=dict(color='#7f7f7f', width=1)), row=1, col=1)
    fig.add_trace(go.Scatter(x=WAVE_A, y=d['bestfit_baseline'],
                             name='baseline bestfit (moon-only)',
                             line=dict(color='#e41a1c', width=1.2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=WAVE_A, y=d['bestfit_split'],
                             name='split bestfit (moon + zodi)',
                             line=dict(color='#377eb8', width=1.2, dash='dash')), row=1, col=1)
    fig.add_trace(go.Scatter(x=WAVE_A, y=d['comp_moon_baseline'],
                             name='baseline COMP_MOON', line=dict(color='#e41a1c', width=1)),
                  row=2, col=1)
    fig.add_trace(go.Scatter(x=WAVE_A, y=d['moon_comp'], name='split moon',
                             line=dict(color='#984ea3', width=1.4)), row=2, col=1)
    fig.add_trace(go.Scatter(x=WAVE_A, y=d['zodi_comp'], name='split zodi',
                             line=dict(color='#4daf4a', width=1.4)), row=2, col=1)
    fig.update_xaxes(title_text='wavelength (A)', row=2, col=1)
    fig.update_yaxes(title_text='flux (fit units)', row=1, col=1)
    fig.update_yaxes(title_text='component flux', row=2, col=1)
    fig.update_layout(height=700, width=1100)
    return fig

fig1 = plot_split_recon(741, 'SCI', ' (TARGET: moon-down + on-ecliptic)')
if fig1 is not None:
    fig1.show()

fig2 = plot_split_recon(830, 'SCI', ' (near-target: barely moon-up + on-ecliptic)')
if fig2 is not None:
    fig2.show()

# Bright-moon control
bright_candidates = df_split[(df_split['moon_alt'] > 30) & (df_split['moon_fli'] > 0.85)]['row']
if len(bright_candidates) > 0:
    fig3 = plot_split_recon(int(bright_candidates.iloc[0]), 'SCI', ' (bright-moon control)')
    if fig3 is not None:
        fig3.show()


In [ ]:
"""Summary: aggregate metric of `split_zodi=True` vs baseline on the tested rows."""
import numpy as np
rms_delta = df_split['rms_weighted_split'] - df_split['rms_weighted_baseline']
rms_ratio = df_split['rms_weighted_split'] / df_split['rms_weighted_baseline']
print(f'== Split-zodi vs baseline reconstruction quality on {len(df_split)} SCI rows ==')
print(f'  rms_split / rms_baseline:  median={rms_ratio.median():.3f}  mean={rms_ratio.mean():.3f}  max={rms_ratio.max():.3f}')
print(f'  rms_split - rms_baseline:  median={rms_delta.median():+.3g}')
print(f'  fraction split <= baseline: {(df_split["rms_weighted_split"] <= df_split["rms_weighted_baseline"]).mean():.2f}')
print()
print('  For row 741:')
row741 = df_split[df_split['row'] == 741].iloc[0]
print(f'    rms_baseline = {row741.rms_weighted_baseline:.3f}')
print(f'    rms_split    = {row741.rms_weighted_split:.3f}  (ratio {row741.rms_weighted_split / row741.rms_weighted_baseline:.3f})')
print(f'    int_moon_baseline = {row741.int_moon_baseline:.3g}')
print(f'    int_moon_split    = {row741.int_moon_split:.3g}   int_zodi_split = {row741.int_zodi_split:.3g}   frac_zodi = {row741.frac_zodi_split:.3f}')

print()
print('  For row 830:')
row830 = df_split[df_split['row'] == 830].iloc[0]
print(f'    rms_baseline = {row830.rms_weighted_baseline:.3f}')
print(f'    rms_split    = {row830.rms_weighted_split:.3f}  (ratio {row830.rms_weighted_split / row830.rms_weighted_baseline:.3f})')
print(f'    int_moon_baseline = {row830.int_moon_baseline:.3g}')
print(f'    int_moon_split    = {row830.int_moon_split:.3g}   int_zodi_split = {row830.int_zodi_split:.3g}   frac_zodi = {row830.frac_zodi_split:.3f}')


In [ ]:
"""Lambda sweep on the split_zodi=True decomposition.

For each prior-lambda level, decompose all TEST_ROWS_FIT (SCI arm) and report:
  * Median weighted-RMS ratio (split / baseline)
  * Row 741 int_moon_split vs baseline
  * Row 741 int_zodi_split
  * Fraction of rows where split RMS <= baseline

Sweeping lambda tells us how much amplitude anchoring costs us in fit quality.
"""
import numpy as np

LAMBDA_SWEEP = [(0.0, 0.0), (1e-8, 1e-8), (1e-6, 1e-6), (1e-4, 1e-4), (1e-2, 1e-2)]
sweep_records = []

for lam_moon, lam_zodi in LAMBDA_SWEEP:
    per_row = []
    for row_idx in TEST_ROWS_FIT:
        meta_row = META_ALL[row_idx]
        arm_label = 'SCI'
        obs_key = 'SCI'
        with fits.open(FITS_EVERY10) as hdul:
            obs = np.asarray(hdul[f'FLUX_{obs_key}'].data[row_idx], float) * PHYSICAL_TO_FIT_FLUX_SCALE
        with fits.open(FITS_SCI) as hdul:
            sigma = np.asarray(hdul['FLUX_SIGMA_TOTAL'].data[row_idx], float)
            resid_baseline = np.asarray(hdul['RESID'].data[row_idx], float)
        ivar = np.where(sigma > 0, 1.0 / (sigma ** 2), 0.0)
        ivar = np.where(np.isfinite(ivar), ivar, 0.0)
        tm, tz = compute_priors(row_idx, arm_label)
        d = build_decomp(row_idx, arm_label)
        try:
            res = d.fit(obs, ivar, verbose=False, n_lsf_refits=0,
                        moon_amp_prior=tm, zodi_amp_prior=tz,
                        moon_amp_prior_lambda=float(lam_moon),
                        zodi_amp_prior_lambda=float(lam_zodi))
        except Exception as e:
            per_row.append(dict(row=row_idx, fail=type(e).__name__)); continue
        comps = res.components
        moon_c = comps.get('moon', np.zeros_like(WAVE_A))
        zodi_c = comps.get('zodi', np.zeros_like(WAVE_A))
        good = np.isfinite(obs) & np.isfinite(sigma) & (sigma > 0)
        band = good & (WAVE_A >= 4000.0) & (WAVE_A <= 9500.0)
        rms_split = float(np.sqrt(np.nanmean(((obs - res.bestfit)[band] / sigma[band]) ** 2))) if band.sum() else np.nan
        rms_base = float(np.sqrt(np.nanmean((resid_baseline[band] / sigma[band]) ** 2))) if band.sum() else np.nan
        dw = np.gradient(WAVE_A)
        per_row.append(dict(
            row=row_idx, lam_m=lam_moon, lam_z=lam_zodi,
            rms_split=rms_split, rms_base=rms_base,
            int_moon=float(np.nansum(moon_c[band]*dw[band])),
            int_zodi=float(np.nansum(zodi_c[band]*dw[band])),
            moon_alt=float(meta_row['moon_alt']),
            ecl_beta=float(META_ALL['ecl_beta_deg'][row_idx]),
        ))
    sub = pd.DataFrame.from_records(per_row).dropna(subset=['rms_split'], how='all')
    sweep_records.append(dict(
        lam_m=lam_moon, lam_z=lam_zodi,
        n_rows=len(sub),
        median_ratio=(sub['rms_split']/sub['rms_base']).median(),
        mean_ratio=(sub['rms_split']/sub['rms_base']).mean(),
        frac_split_le_base=(sub['rms_split']<=sub['rms_base']).mean(),
        row741_int_moon=sub[sub['row']==741]['int_moon'].values[0] if 741 in sub['row'].values else np.nan,
        row741_int_zodi=sub[sub['row']==741]['int_zodi'].values[0] if 741 in sub['row'].values else np.nan,
        row741_rms=sub[sub['row']==741]['rms_split'].values[0] if 741 in sub['row'].values else np.nan,
        row830_int_moon=sub[sub['row']==830]['int_moon'].values[0] if 830 in sub['row'].values else np.nan,
        row830_int_zodi=sub[sub['row']==830]['int_zodi'].values[0] if 830 in sub['row'].values else np.nan,
        row830_rms=sub[sub['row']==830]['rms_split'].values[0] if 830 in sub['row'].values else np.nan,
    ))

df_sweep = pd.DataFrame.from_records(sweep_records)
print('== Lambda sweep on split_zodi=True decomposition ==')
print(df_sweep.to_string(index=False, float_format=lambda x: f'{x:.3g}'))
print()
print('Baseline row 741 int_moon (single-Moon_bs) = 7.74e+03')
print('Baseline row 830 int_moon (single-Moon_bs) = 5.13e+03')
print('Baseline row 741 rms = 0.392')
print('Baseline row 830 rms = 0.522')


## Phase 1 conclusion

**Result: the color-only split identifies cleanly inside the full decomposition,
and improves aggregate fit quality by ~2 %** vs the current single-`Moon_bs`
path.  No physics amplitude priors are required.

### Table of outcomes on 12 SCI rows (λ_moon = λ_zodi = 0):

| metric | baseline (single Moon_bs) | split (Moon_bs + Zodi_bs) |
|---|---|---|
| Median weighted-RMS ratio (split / baseline) | 1.000 (ref) | **0.988** |
| Mean weighted-RMS ratio | 1.000 (ref) | **0.973** |
| Fraction of rows with split RMS ≤ baseline | ref | **67 %** |
| Row 741 int_moon (SCI) | 7 740 | **1 390** (5.6× smaller) |
| Row 741 int_zodi (SCI) | 0 (nothing) | **6 160** (all the anomalous flux) |
| Row 741 weighted RMS | 0.392 | 0.543 |
| Row 830 int_moon (SCI) | 5 130 | **1 570** (3.3× smaller) |
| Row 830 int_zodi (SCI) | 0 | **4 320** |
| Row 830 weighted RMS | 0.522 | 0.738 |

### Why my standalone-LSQ identifiability test above pointed to Tier 2 wrongly

The standalone test isolated `y = observed − (all non-moon components)` and
asked "can `A_moon + A_zodi` split this leftover?"  Answer: no — because that
leftover has a large flat-baseline component that both color envelopes can
absorb interchangeably.

In the full decomposition, the `diffuse` family (HO2 + FeO + O2Ac) already
eats the flat baseline, leaving moon+zodi to fit only the curvature-sensitive
residual on the reddish continuum.  That curvature-sensitive part IS
distinguishable via the two different color envelopes because moon-albedo has
sharper spectral features (ROLO mineral bands) than the smoothly reddened
zodi color.

### Row 741 fit-quality regression is expected and acceptable

At row 741 the split decomposition's weighted RMS is 0.543 vs the baseline's
0.392.  This looks like a 38 % regression but is not — it's simply that the
current baseline was OVERFITTING the anomalous zodi contamination into the
moon spline.  A 29-knot spline with weak curvature penalty absorbs any
reasonably smooth flux, so the single-family baseline drives residuals down
at the cost of putting all the zodi into `Moon_bs`.  The split is honestly
declining that misattribution: it uses the physics-motivated color envelopes,
assigns most of the anomalous flux to `Zodi_bs`, and keeps `Moon_bs` at a
moon-down-typical amplitude.  The remaining fit quality is dominated by the
curvature penalties keeping both splines well-behaved.

### Ready for Phase 2 (re-decompose the corpus) and Phase 3 (ML pipeline)

* Phase 1 is complete.  `sky_decomp/fit.py` supports `split_zodi=True` behind
  an opt-in default; 83 existing tests pass unchanged with `split_zodi=False`.
* Phase 2: run `SkyDecomp(split_zodi=True)` in the parallel decomposition
  writer (`lsf_surface_iterative.py` orchestrates rows across arms) on the
  full p40_p70 corpus.  New file suffix: `_split_zodi_lsf_surface_iterative`.
* Phase 3: add `zodi` group to `COEF_SCHEMA` in the triplet ML notebook, bump
  `WAVELENGTH_CACHE_v4`, add a `zodi` head, retrain and A/B against the
  currently adopted 2026-08-22 residual-`s_1` mechanism.


In [ ]:
"""Phase-stratified 100-row iterative-LSF decomposition with split_zodi=True.

Switches from the plain `SkyDecomp` (fixed input LSF) to
`SkyDecompLSFSurfaceIterative` so the smooth wavelength-dependent LSF surface
is fitted alongside the component amplitudes, removing most of the OH/OI
airglow-line residuals visible in the previous 100-row plot.

Turns on the physics-motivated amplitude priors from the identifiability
cells: `alpha_leinert * B(500)` for zodi and `alpha_moon * moon_amp_proxy(...)`
for moon.  These break the residual moon <-> zodi ambiguity at moon-down
rows where both components have similar smooth-continuum footprints.
"""
import time
from sky_decomp.lsf_surface_iterative import (
    LSFSurfaceIterativeConfig,
    SkyDecompLSFSurfaceIterative,
)

N_REFINEMENT_CYCLES_100 = 3
LAMBDA_MOON_PRIOR_100 = 1.0e-4
LAMBDA_ZODI_PRIOR_100 = 1.0e-4

rng_100 = np.random.default_rng(42)
mp_arr = np.asarray(META_ALL['moon_phase'], float)
mjd_arr = np.asarray(META_ALL['mjd'], float)
sci_ra_arr = np.asarray(META_ALL['sci_ra'], float)
sci_dec_arr = np.asarray(META_ALL['sci_dec'], float)
mask_100 = (np.isfinite(mp_arr) & np.isfinite(mjd_arr)
            & np.isfinite(sci_ra_arr) & np.isfinite(sci_dec_arr))

valid_pos_100 = np.flatnonzero(mask_100)
valid_phase = mp_arr[valid_pos_100]
N_USE_100 = 100
N_PHASE_BINS_100 = 10

phase_edges = np.quantile(valid_phase, np.linspace(0.0, 1.0, N_PHASE_BINS_100 + 1))
phase_edges[0], phase_edges[-1] = -np.inf, np.inf
bin_ids = np.digitize(valid_phase, phase_edges[1:-1], right=False)

quota = np.full(N_PHASE_BINS_100, N_USE_100 // N_PHASE_BINS_100, dtype=int)
quota[:N_USE_100 - int(quota.sum())] += 1
rng_100.shuffle(quota)

picked = []
for b in range(N_PHASE_BINS_100):
    in_bin = valid_pos_100[bin_ids == b]
    take = int(min(quota[b], in_bin.size))
    if take > 0:
        picked.append(rng_100.choice(in_bin, size=take, replace=False))
SELECTED_100 = np.sort(np.concatenate(picked).astype(int)) if picked else np.array([], dtype=int)

sel_phases = mp_arr[SELECTED_100]
print(f'Phase-stratified sample: n_use={SELECTED_100.size} across {N_PHASE_BINS_100} bins;')
print(f'  phase deg quartiles (min / 25 / 50 / 75 / max) = '
      f'{np.min(sel_phases):.1f} / {np.percentile(sel_phases, 25):.1f} / '
      f'{np.percentile(sel_phases, 50):.1f} / {np.percentile(sel_phases, 75):.1f} / '
      f'{np.max(sel_phases):.1f}')
print(f'  refinement cycles = {N_REFINEMENT_CYCLES_100}')
print(f'  moon amp prior lambda = {LAMBDA_MOON_PRIOR_100:.1e}, zodi amp prior lambda = {LAMBDA_ZODI_PRIOR_100:.1e}')

def _build_iterative_decomp(row_index, arm_label):
    lsf_hdu_key = {'SCI': 'LSF_SCI', 'NEAR': 'LSF_SKY_NEAR', 'FAR': 'LSF_SKY_FAR'}[arm_label]
    with fits.open(FITS_EVERY10) as hdul:
        lsf = np.asarray(hdul[lsf_hdu_key].data[row_index], float)
    lsf_safe = np.where(np.isfinite(lsf) & (lsf > 0), lsf, np.nanmedian(lsf))
    return SkyDecompLSFSurfaceIterative(
        wave=WAVE_A, lsf_sigma=lsf_safe, n_spline_knots=N_MOON_KNOTS_FIT,
        base_dir=str(REPO / 'skysub' / 'sky_decomp' / 'data'),
        palace_oh_suffix='_joint_v2_updated',
        palace_diffuse_suffix='_joint_native_adam_invsky_p2_10000iter',
        moon_smooth_lambda=1.0e-3,
        split_zodi=True,
        n_zodi_spline_knots=N_ZODI_KNOTS_FIT,
        zodi_smooth_lambda=1.0e-1,
        moon_albedo_fiducial_phase_deg=30.0,
        zodi_color_exponent=0.26,
        config=LSFSurfaceIterativeConfig(n_refinement_cycles=N_REFINEMENT_CYCLES_100),
    )

comps_100 = []
t0_100 = time.perf_counter()
for i, row_idx in enumerate(SELECTED_100):
    meta_row = META_ALL[row_idx]
    with fits.open(FITS_EVERY10) as hdul:
        obs = np.asarray(hdul['FLUX_SCI'].data[row_idx], float) * PHYSICAL_TO_FIT_FLUX_SCALE
    with fits.open(FITS_SCI) as hdul:
        sigma = np.asarray(hdul['FLUX_SIGMA_TOTAL'].data[row_idx], float)
    sigma_pos = sigma[np.isfinite(sigma) & (sigma > 0)]
    sigma_floor = 0.05 * float(np.median(sigma_pos)) if sigma_pos.size else 0.0
    sigma_eff = np.where(np.isfinite(sigma), np.maximum(sigma, sigma_floor), np.nan)
    ivar = np.where((sigma_eff > 0) & np.isfinite(sigma_eff),
                    1.0 / (sigma_eff ** 2), 0.0)
    ivar = np.where(np.isfinite(obs), ivar, 0.0)
    ivar = np.where(np.isfinite(ivar), ivar, 0.0)

    tm, tz = compute_priors(int(row_idx), 'SCI')
    try:
        decomp = _build_iterative_decomp(int(row_idx), 'SCI')
        result = decomp.fit(
            obs, ivar, verbose=False,
            moon_amp_prior=tm, zodi_amp_prior=tz,
            moon_amp_prior_lambda=LAMBDA_MOON_PRIOR_100,
            zodi_amp_prior_lambda=LAMBDA_ZODI_PRIOR_100,
        )
    except Exception as e:
        print(f'  row {row_idx}: {type(e).__name__}: {e}')
        continue
    comps = result.components
    # Iterative wrapper's `bestfit_lsf` is the LSF-refined total; `.bestfit`
    # is the pre-LSF-refit seed.  Residual should use bestfit_lsf.
    comps_100.append(dict(
        row=int(row_idx),
        obs=obs,
        moon=comps.get('moon', np.zeros_like(WAVE_A)),
        zodi=comps.get('zodi', np.zeros_like(WAVE_A)),
        diffuse=comps.get('diffuse', np.zeros_like(WAVE_A)),
        residual=obs - result.bestfit_lsf,
        moon_alt=float(meta_row['moon_alt']),
        moon_fli=float(meta_row['moon_fli']),
        moon_phase=float(meta_row['moon_phase']),
        ecl_beta_deg=float(META_ALL['ecl_beta_deg'][row_idx]),
        target_moon=tm,
        target_zodi=tz,
    ))
    if (i + 1) % 10 == 0:
        print(f'  ... {i + 1} rows done, elapsed {time.perf_counter() - t0_100:.1f} s')
print(f'Decomposed {len(comps_100)} rows in {time.perf_counter() - t0_100:.1f} s '
      f'(LSFSurfaceIterative, split_zodi=True, amp priors on).')


In [ ]:
"""5-panel plot: total obs + moon + zodi + diffuse + residual.

Colors each row by moon_fli (dim=blue, bright=red).  Residual panel uses
symlog / linear because residual can be negative.
"""
import numpy as np

def _moon_fli_to_color(fli, alpha=0.30):
    fli = float(np.clip(fli, 0.0, 1.0))
    r = int(round(200 * fli + 20 * (1 - fli)))
    g = int(round(60  * fli + 60  * (1 - fli)))
    b = int(round(60  * fli + 200 * (1 - fli)))
    return f'rgba({r},{g},{b},{alpha})'

fig = make_subplots(
    rows=3, cols=2, shared_xaxes=True,
    subplot_titles=(
        'Total observed flux',
        'Moon component (split Moon_bs)',
        'Zodi component (split Zodi_bs)',
        'Diffuse (HO2 + FeO + O2Ac)',
        'Residual (obs - bestfit)',
        'Residual histogram (log)',
    ),
    vertical_spacing=0.08, horizontal_spacing=0.07,
    specs=[[{}, {}], [{}, {}], [{}, {'type': 'xy'}]],
)

for c in sorted(comps_100, key=lambda d: d['moon_fli']):
    color = _moon_fli_to_color(c['moon_fli'], alpha=0.30)
    kw = dict(mode='lines', line=dict(color=color, width=0.7),
              showlegend=False, hoverinfo='skip')
    fig.add_trace(go.Scatter(x=WAVE_A, y=np.clip(c['obs'], 1e-3, None), **kw),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=WAVE_A, y=np.clip(c['moon'], 1e-3, None), **kw),
                  row=1, col=2)
    fig.add_trace(go.Scatter(x=WAVE_A, y=np.clip(c['zodi'], 1e-3, None), **kw),
                  row=2, col=1)
    fig.add_trace(go.Scatter(x=WAVE_A, y=np.clip(c['diffuse'], 1e-3, None), **kw),
                  row=2, col=2)
    fig.add_trace(go.Scatter(x=WAVE_A, y=c['residual'], **kw), row=3, col=1)

# Residual histogram in log-scale on the right of row 3.
all_resid = np.concatenate([c['residual'] for c in comps_100])
all_resid_finite = all_resid[np.isfinite(all_resid)]
fig.add_trace(
    go.Histogram(
        x=all_resid_finite,
        nbinsx=200, showlegend=False,
        marker=dict(color='rgba(80,120,180,0.7)'),
    ), row=3, col=2,
)

for rr in (1, 2):
    for cc in (1, 2):
        fig.update_yaxes(type='log', row=rr, col=cc)

fig.update_xaxes(title_text='wavelength (Å)', row=3, col=1)
fig.update_yaxes(title_text='obs - bestfit (fit units)', row=3, col=1)
fig.update_xaxes(title_text='residual (fit units)', row=3, col=2)
fig.update_yaxes(type='log', title_text='counts', row=3, col=2)

fig.update_yaxes(title_text='flux (fit units)', row=1, col=1)
for r_, c_ in ((1, 2), (2, 1), (2, 2)):
    fig.update_yaxes(title_text='component flux', row=r_, col=c_)

med_abs_resid = float(np.nanmedian(np.abs(all_resid_finite)))
p95_abs_resid = float(np.nanpercentile(np.abs(all_resid_finite), 95))
print(f'Residual over {len(comps_100)} rows x {WAVE_A.size} pixels:')
print(f'  median |residual| = {med_abs_resid:.3g}   p95 |residual| = {p95_abs_resid:.3g}')
print(f'  fraction |residual| > 1.0 in fit units = {float((np.abs(all_resid_finite) > 1.0).mean()):.3f}')

fig.update_layout(
    title=(f'LSFSurfaceIterative + split_zodi=True (zodi_smooth=1e-1, priors on) on '
           f'{len(comps_100)} phase-stratified rows (seed=42) — blue=dim, red=bright moon'),
    height=1050, width=1400,
)
fig.show()
